In [1]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

sys.path.insert(0, str(Path.home() / "Desktop/Proyectos/2026/02_Libreria_multimodal/multimodalpy"))
from multimodalpy import main


NAP_API_KEY= "44a2e8d8-fab3-4d87-940d-d7788a7dac94"

output_folder = Path.home() / "Desktop/Proyectos/2026/02_Libreria_multimodal/multimodalpy/multimodalpy/data/outputs/output_Valencia"
boundaries = Path.home() / "Desktop/Proyectos/2026/02_Libreria_multimodal/multimodalpy/multimodalpy/data/tests/boundaries/capa_Valencia.shp"

AREA_NAME = "Valencia"
CRS = "epsg:25830"
MODES = ["driving", "walking", "bike", "bus", "train"]


In [2]:
from network_qa import (
    run_all_checks_osm,
    results_to_frame,
    style_results,
    summary_matrix,
    style_summary_matrix,
    show_issues,
)


In [ ]:
STEM_BY_MODE = {
    "driving": "osm_driving",
    "walking": "osm_walking",
    "bike": "osm_bike",
    "train": "gtfs_20260902_080005_metro_valencia",
    "bus": "gtfs_20260908_020020_genvalenciana_interurbano",
}


def load_network(mode: str, fmt: str, gpkg_path: Path | None = None):
    """Devuelve (nodes, edges) para `mode` en el formato `fmt`.

    fmt: "geojson" | "shapefile" | "geopackage"
    """
    stem = STEM_BY_MODE[mode]

    if fmt == "geojson":
        nodes = gpd.read_file(output_folder / f"{stem}_nodes.geojson")
        edges = gpd.read_file(output_folder / f"{stem}_edges.geojson")
    elif fmt == "shapefile":
        nodes = gpd.read_file(output_folder / f"{stem}_nodes.shp")
        edges = gpd.read_file(output_folder / f"{stem}_edges.shp")
    elif fmt == "geopackage":
        nodes = gpd.read_file(gpkg_path, layer=f"{stem}_nodes")
        edges = gpd.read_file(gpkg_path, layer=f"{stem}_edges")
    else:
        raise ValueError(f"Formato desconocido: {fmt!r}")

    return nodes, edges


def run_checks_for_all_modes(fmt: str, gpkg_path: Path | None = None) -> dict:
    """Corre run_all_checks_osm para cada modo en MODES, en el formato dado.

    Devuelve {modo: [CheckResult, ...]} — usa `show_results()` (celda de
    abajo) para verlo como tabla en vez de leer el diccionario a mano.
    """
    results_by_mode = {}
    for mode in MODES:
        nodes, edges = load_network(mode, fmt, gpkg_path=gpkg_path)
        results_by_mode[mode] = run_all_checks_osm(nodes, edges, verbose=False)
    return results_by_mode


In [4]:
def show_results(results_by_mode: dict):
    matrix = summary_matrix(results_by_mode)
    return style_summary_matrix(matrix)


## GeoJSON

In [ ]:
main(
    area_name=AREA_NAME,
    modes=MODES,
    output_path=str(output_folder),
    crs=CRS,
    output_file_type="geojson",
    boundaries_path=str(boundaries),
    api_key=NAP_API_KEY,
)

results_geojson = run_checks_for_all_modes("geojson")
show_results(results_geojson)


In [ ]:
print(show_issues(results_geojson["bus"], "nodes: expected columns present"))
print(show_issues(results_geojson["train"], "no duplicate (edge_id, from_node_id, to_node_id) rows"))

## Shapefile

In [ ]:
main(
    area_name=AREA_NAME,
    modes=MODES,
    output_path=str(output_folder),
    crs=CRS,
    output_file_type="shapefile",
    boundaries_path=str(boundaries),
    api_key=NAP_API_KEY,
)

results_shapefile = run_checks_for_all_modes("shapefile")
show_results(results_shapefile)


## Geopackage

In [ ]:
main(
    area_name=AREA_NAME,
    modes=MODES,
    output_path=str(output_folder),
    crs=CRS,
    output_file_type="geopackage",
    boundaries_path=str(boundaries),
    api_key=NAP_API_KEY,
)

gpkg_path = output_folder / "valencia.gpkg"
results_geopackage = run_checks_for_all_modes("geopackage", gpkg_path=gpkg_path)
show_results(results_geopackage)
